# Домашнее задание №6

## Детекция объекта внимания в видеопотоке

### Цель

Построить воспроизводимый конвейер выделения движущегося объекта на видео со статичной камеры и измерить, как параметры модели фона влияют на пропуски и ложные срабатывания.

Результатом работы является количественная оценка на размеченном фрагменте и вывод о том, при каких условиях модель фона перестаёт работать, а не подобранный под один ролик набор параметров.

[Методические указания блока](README.md) · [Общие МУ](../../../docs/guidelines-students.md) · [Рубрика оценивания](../teachers-assessment/README.md)

## 1. Что используется в работе

Библиотеки: `opencv-python`, `numpy`, `scikit-image`, `matplotlib`, `pandas`.

Данные. Ноутбук работает без интернета: видео генерируется синтетически. Генератор даёт для каждого кадра эталонный прямоугольник объекта и эталонную маску переднего плана, поэтому полнота и точность считаются, а не описываются словами. В синтетику специально заложены:

- статичный текстурированный фон и аддитивный шум камеры;
- мелкий колеблющийся объект-помеха (источник ложных срабатываний);
- скачок освещения в середине последовательности (проверка адаптации модели фона).

Заготовка содержит также функцию чтения реального видеофайла: если вы подложите собственную съёмку со статичной камеры или последовательность MOT17 (карточки в [resources/datasets](../../resources/datasets/README.md)), остальной код не меняется. Для реального видео эталонные боксы придётся разметить самостоятельно хотя бы на коротком фрагменте.

Готовыми даны: генератор видео, чтение и запись видеофайлов, метрики по кадрам и по пикселям, выделение наибольшей компоненты, разбиение на настроечный и оценочный фрагменты, журнал экспериментов. Модели фона, морфологическая постобработка, критерий устойчивости объекта внимания и планирование серий — ваша часть.

In [ ]:
# Зависимости (при необходимости раскомментируйте):
# %pip install opencv-python numpy scikit-image matplotlib pandas

import platform
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import skimage

SEED = 42
rng = np.random.default_rng(SEED)
cv2.setRNGSeed(SEED)

OUTPUT_DIR = Path("outputs_hw6")
(OUTPUT_DIR / "figures").mkdir(parents=True, exist_ok=True)

print("python      :", platform.python_version())
print("opencv      :", cv2.__version__)
print("numpy       :", np.__version__)
print("scikit-image:", skimage.__version__)
print("pandas      :", pd.__version__)
print("SEED        :", SEED)

## 2. Краткая теоретическая справка

### 2.1. Разностные методы

Простейшая модель фона — предыдущий кадр или скользящее среднее:

$$B_t = \alpha I_t + (1-\alpha) B_{t-1}, \qquad
F_t(x,y) = \bigl[\,\lvert I_t(x,y) - B_t(x,y) \rvert > T \,\bigr].$$

Коэффициент $\alpha$ задаёт скорость адаптации. Малое $\alpha$ — медленная адаптация и длинный «шлейф» после изменения сцены; большое $\alpha$ — объект, остановившийся на несколько кадров, растворяется в фоне.

### 2.2. Смесь гауссиан (MOG2)

Каждый пиксель описывается смесью $K$ нормальных распределений:

$$P(I_t) = \sum_{i=1}^{K} \omega_i\, \mathcal{N}\bigl(I_t;\ \mu_i,\ \Sigma_i\bigr), \qquad \sum_i \omega_i = 1 .$$

Пиксель относится к фону, если попадает в одну из компонент, признанных фоновыми (набравших суммарный вес выше порога). Веса и параметры обновляются рекуррентно со скоростью обучения `learningRate`. Многомодальность позволяет описывать периодические изменения (колеблющаяся ветка, мерцание), недоступные одномодальной разностной схеме.

Опасность адаптации: при плавном изменении освещения модель подстраивается и сохраняет полноту, но медленно движущийся или остановившийся объект она поглощает как фон. При скачке освещения весь кадр на некоторое время становится «передним планом». Этот компромисс необходимо измерить.

### 2.3. Морфологическая очистка

Открытие $ (F \ominus S) \oplus S $ удаляет мелкие ложные отклики, закрытие $ (F \oplus S) \ominus S $ заполняет разрывы внутри объекта. Размер и форма структурирующего элемента $S$ — исследуемые параметры, а не константы «по умолчанию».

### 2.4. Метрики

Для покадровой оценки детекции применяется порог по IoU прямоугольников:

$$\mathrm{IoU}(A,B) = \frac{\lvert A \cap B \rvert}{\lvert A \cup B \rvert}, \qquad
\mathrm{TP}: \mathrm{IoU} \ge \tau,\ \ \tau = 0.5 .$$

$$\mathrm{Precision} = \frac{TP}{TP+FP}, \qquad \mathrm{Recall} = \frac{TP}{TP+FN}, \qquad
F_1 = \frac{2\,PR}{P+R}.$$

Дополнительно считается попиксельная оценка маски переднего плана — она чувствительнее к качеству морфологии, чем оценка по прямоугольникам.

## 3. Задачи

Формулировка из [методических указаний блока](README.md#дз6-детекция-объекта-внимания-в-видеопотоке):

Для видеоролика со статичной камерой:

1. Организуйте покадровое чтение и буферизацию.
2. Выделите движущиеся объекты методами вычитания фона (разностные кадры, MOG2/KNN).
3. Очистите маски морфологией и выделите объект внимания (наибольшая устойчивая компонента).
4. Исследуйте влияние параметров модели фона на пропуски и ложные срабатывания.

**Результат:** видео/серия кадров с выделенным объектом; количественная оценка на размеченном фрагменте (полнота/точность по кадрам).

Проверяемые элементы ([рубрика](../teachers-assessment/README.md)): модель фона сравнена минимум с разностным методом; маски очищены морфологией; полнота/точность посчитаны на размеченном фрагменте, а не описаны словесно.

## 4. Данные: синтетическое видео с эталоном

Объект — светлый прямоугольник, движущийся по синусоидальной траектории. Помеха — мелкий объект, колеблющийся на месте: разностный метод реагирует на него так же, как на объект внимания, многомодальная модель фона — нет. Скачок освещения на заданном кадре имитирует включение света.

Все факторы задаются параметрами, поэтому вы можете строить контролируемые серии не только по параметрам метода, но и по свойствам сцены.

In [ ]:
def make_background(h, w, seed=SEED):
    """Статичный текстурированный фон.

    Выход: (h, w, 3) uint8, BGR.
    """
    generator = np.random.default_rng(seed)
    coarse = generator.normal(0.0, 1.0, (max(h // 8, 2), max(w // 8, 2), 3))
    field = cv2.resize(coarse, (w, h), interpolation=cv2.INTER_CUBIC)
    field = cv2.GaussianBlur(field, (0, 0), 3)
    field = 40.0 + 60.0 * (field - field.min()) / (float(np.ptp(field)) + 1e-9)
    gradient = np.linspace(0.0, 50.0, w, dtype=np.float32)[None, :, None]
    return np.clip(field + gradient, 0, 255).astype(np.uint8)


def make_synthetic_video(n_frames=150, size=(240, 320), *, noise_sigma=6.0,
                         object_size=(46, 34), distractor=True,
                         illumination=None, occluder=None, seed=SEED):
    """Синтетическое видео со статичной камеры и эталонной разметкой.

    Вход:
        n_frames     — число кадров;
        size         — (h, w);
        noise_sigma  — СКО шума камеры;
        object_size  — (ширина, высота) объекта внимания;
        distractor   — добавить колеблющуюся помеху;
        illumination — None или (frame_index, gain): скачок освещения;
        occluder     — None или (x, y, w, h): статичная перекрывающая область.
    Выход:
        frames — список (h, w, 3) uint8 BGR;
        masks  — список (h, w) uint8 {0, 1}: эталонная маска объекта внимания;
        gt     — DataFrame с колонками frame, x, y, w, h, visible.
    """
    h, w = size
    generator = np.random.default_rng(seed)
    background = make_background(h, w, seed)
    ow, oh = object_size

    frames, masks, rows = [], [], []
    for index in range(n_frames):
        t = index / max(n_frames - 1, 1)
        cx = 30.0 + (w - 60.0) * t
        cy = h * 0.5 + 0.22 * h * np.sin(2.0 * np.pi * 1.5 * t)
        x = int(np.clip(cx - ow / 2.0, 0, w - ow))
        y = int(np.clip(cy - oh / 2.0, 0, h - oh))

        frame = background.copy()
        if distractor:
            shift = int(round(6.0 * np.sin(2.0 * np.pi * 6.0 * t)))
            cv2.circle(frame, (int(0.12 * w) + shift, int(0.85 * h)), 7, (160, 160, 160), -1)

        mask = np.zeros((h, w), dtype=np.uint8)
        cv2.rectangle(mask, (x, y), (x + ow, y + oh), 1, -1)
        frame[mask == 1] = (225, 225, 235)

        visible = 1
        if occluder is not None:
            ox, oy, oww, ohh = occluder
            cv2.rectangle(frame, (ox, oy), (ox + oww, oy + ohh), (35, 35, 45), -1)
            inter_w = max(0, min(x + ow, ox + oww) - max(x, ox))
            inter_h = max(0, min(y + oh, oy + ohh) - max(y, oy))
            covered = (inter_w * inter_h) / float(ow * oh)
            mask[oy:oy + ohh, ox:ox + oww] = 0
            visible = int(covered < 0.6)

        frame = frame.astype(np.float32)
        if illumination is not None and index >= illumination[0]:
            frame = frame * float(illumination[1])
        frame = frame + generator.normal(0.0, noise_sigma, frame.shape)
        frames.append(np.clip(frame, 0, 255).astype(np.uint8))
        masks.append(mask)
        rows.append({"frame": index, "x": x, "y": y, "w": ow, "h": oh, "visible": visible})

    return frames, masks, pd.DataFrame(rows)


frames, gt_masks, gt = make_synthetic_video(illumination=(90, 1.35))
print("Кадров:", len(frames), "| размер:", frames[0].shape)
display(gt.head())

In [ ]:
def show_frames(frames, indices, boxes=None, title=""):
    """Показать несколько кадров с прямоугольниками.

    Вход: boxes — dict {frame_index: (x, y, w, h)} или None.
    """
    fig, axes = plt.subplots(1, len(indices), figsize=(3.2 * len(indices), 3.4))
    for ax, index in zip(np.atleast_1d(axes), indices):
        canvas = frames[index].copy()
        if boxes and index in boxes and boxes[index] is not None:
            x, y, w, h = [int(v) for v in boxes[index]]
            cv2.rectangle(canvas, (x, y), (x + w, y + h), (0, 255, 0), 2)
        ax.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
        ax.set_title(f"кадр {index}", fontsize=9)
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def read_video_frames(path, max_frames=None, to_size=None):
    """Покадровое чтение видеофайла с буферизацией в список.

    Вход:  path — путь к файлу; max_frames — ограничение; to_size — (w, h) или None.
    Выход: список кадров (H, W, 3) uint8 BGR.

    Для длинных роликов не буферизуйте весь файл: обрабатывайте кадры потоково,
    сохраняя только маски и прямоугольники.
    """
    capture = cv2.VideoCapture(str(path))
    if not capture.isOpened():
        raise FileNotFoundError(f"Не удалось открыть видео: {path}")
    buffer = []
    try:
        while max_frames is None or len(buffer) < max_frames:
            ok, frame = capture.read()
            if not ok:
                break
            if to_size is not None:
                frame = cv2.resize(frame, to_size, interpolation=cv2.INTER_AREA)
            buffer.append(frame)
    finally:
        capture.release()
    return buffer


def save_video(frames, path, fps=25):
    """Записать последовательность кадров в файл (для приложения к отчёту)."""
    h, w = frames[0].shape[:2]
    writer = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    for frame in frames:
        writer.write(frame)
    writer.release()
    return path


gt_boxes = {int(r.frame): (r.x, r.y, r.w, r.h) for r in gt.itertuples() if r.visible}
show_frames(frames, [0, 45, 90, 120, 149], gt_boxes, "Синтетическое видео и эталонные боксы")

## 5. Разбиение на настроечный и оценочный фрагменты

Типичная ошибка ([рубрика](../teachers-assessment/README.md#типичные-ошибки)): параметры подобраны на том же фрагменте, на котором посчитаны метрики. Итоговые числа в этом случае оптимистичны и не переносятся на новые данные.

Правило работы: все серии по параметрам выполняются на настроечном фрагменте, финальные метрики считаются один раз на оценочном фрагменте после фиксации конфигурации.

Разбиение по времени (а не случайное) обязательно: соседние кадры почти идентичны, случайное разбиение приведёт к утечке.

In [ ]:
def split_frames(frames, masks, gt, tune_ratio=0.5):
    """Разбить последовательность по времени на настроечную и оценочную части.

    Выход: словарь с ключами 'tune' и 'eval'; каждое значение — кортеж
           (frames, masks, gt_frame) с нумерацией кадров, начинающейся с нуля.
    """
    split_at = int(len(frames) * tune_ratio)
    parts = {}
    for name, (start, stop) in {"tune": (0, split_at), "eval": (split_at, len(frames))}.items():
        part_gt = gt[(gt["frame"] >= start) & (gt["frame"] < stop)].copy()
        part_gt["frame"] = part_gt["frame"] - start
        parts[name] = (frames[start:stop], masks[start:stop], part_gt.reset_index(drop=True))
    return parts


parts = split_frames(frames, gt_masks, gt, tune_ratio=0.5)
for name, (part_frames, _, part_gt) in parts.items():
    print(f"{name}: кадров {len(part_frames)}, видимых объектов {int(part_gt['visible'].sum())}")

# Скачок освещения приходится на оценочный фрагмент. Это осознанное решение:
# оно проверяет перенос настроек на изменившиеся условия. Если вы хотите
# исследовать адаптацию отдельно, сгенерируйте вторую последовательность
# со скачком в настроечной части и укажите это в отчёте.

## 6. Метрики и журнал экспериментов

Считаются два уровня оценки:

- **по кадрам**: TP/FP/FN с порогом IoU 0.5 по прямоугольнику объекта внимания;
- **по пикселям**: precision/recall/IoU маски переднего плана.

Кадры, где объект не виден (`visible = 0`), исключаются из знаменателя полноты, но ложное срабатывание на них по-прежнему штрафуется.

In [ ]:
def iou_box(box_a, box_b):
    """IoU двух прямоугольников формата (x, y, w, h)."""
    if box_a is None or box_b is None:
        return 0.0
    ax, ay, aw, ah = box_a
    bx, by, bw, bh = box_b
    inter_w = max(0.0, min(ax + aw, bx + bw) - max(ax, bx))
    inter_h = max(0.0, min(ay + ah, by + bh) - max(ay, by))
    inter = inter_w * inter_h
    union = aw * ah + bw * bh - inter
    return 0.0 if union <= 0 else float(inter / union)


def evaluate_boxes(gt_frame, predictions, iou_threshold=0.5):
    """Покадровая точность и полнота детекции объекта внимания.

    Вход:
        gt_frame    — DataFrame с колонками frame, x, y, w, h, visible;
        predictions — dict {frame_index: (x, y, w, h) или None};
        iou_threshold — порог засчитывания детекции.
    Выход: словарь с TP, FP, FN, precision, recall, f1, mean_iou.
    """
    tp = fp = fn = 0
    ious = []
    for row in gt_frame.itertuples():
        predicted = predictions.get(int(row.frame))
        truth = (row.x, row.y, row.w, row.h) if row.visible else None
        value = iou_box(truth, predicted) if (truth and predicted) else 0.0
        if truth is not None:
            ious.append(value)
        if truth is not None and predicted is not None and value >= iou_threshold:
            tp += 1
        else:
            if predicted is not None:
                fp += 1
            if truth is not None:
                fn += 1
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"TP": tp, "FP": fp, "FN": fn,
            "precision": round(precision, 4), "recall": round(recall, 4),
            "f1": round(f1, 4), "mean_iou": round(float(np.mean(ious)) if ious else 0.0, 4)}


def evaluate_masks(true_masks, predicted_masks):
    """Попиксельная оценка маски переднего плана по всей последовательности."""
    truth = np.concatenate([m.astype(bool).ravel() for m in true_masks])
    predicted = np.concatenate([m.astype(bool).ravel() for m in predicted_masks])
    tp = int(np.logical_and(truth, predicted).sum())
    fp = int(np.logical_and(~truth, predicted).sum())
    fn = int(np.logical_and(truth, ~predicted).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    iou = tp / (tp + fp + fn) if tp + fp + fn else 0.0
    return {"px_precision": round(precision, 4), "px_recall": round(recall, 4),
            "px_iou": round(iou, 4)}


def largest_component_box(mask, min_area=150):
    """Прямоугольник наибольшей связной компоненты маски.

    Вход:  mask (H, W) uint8 {0, 1}; min_area — порог отбрасывания мелких компонент.
    Выход: (x, y, w, h) или None.

    Замечание: это выбор наибольшей компоненты в одном кадре. Требование задания —
    наибольшая УСТОЙЧИВАЯ компонента, то есть согласованная по времени; критерий
    устойчивости реализуете вы (раздел 8).
    """
    count, _, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), 8)
    best, best_area = None, min_area
    for index in range(1, count):
        x, y, w, h, area = stats[index]
        if area > best_area:
            best, best_area = (int(x), int(y), int(w), int(h)), area
    return best


RUNS = []


def log_run(**fields):
    """Добавить запись в журнал экспериментов."""
    record = {"run_id": len(RUNS), **fields}
    RUNS.append(record)
    return record


def runs_table(columns=None, sort_by=None):
    if not RUNS:
        return pd.DataFrame()
    frame = pd.DataFrame(RUNS)
    if columns:
        frame = frame[[c for c in columns if c in frame.columns]]
    if sort_by:
        frame = frame.sort_values(sort_by)
    return frame.reset_index(drop=True)


def save_runs(path=OUTPUT_DIR / "runs.csv"):
    runs_table().to_csv(path, index=False)
    return path

## 7. Baseline: разностный метод

Опорная конфигурация — вычитание статичной модели фона, построенной как медиана первых кадров, с фиксированным порогом и без морфологии. Baseline не изменяется после начала основной серии: все последующие варианты сравниваются именно с ним.

Ниже конвейер реализован полностью — это «рельсы». Убедитесь, что понимаете каждый шаг, прежде чем расширять его моделями фона.

In [ ]:
def run_frame_difference(frames, *, threshold=25, warmup=15, min_area=150):
    """Опорный метод: |кадр - медианный фон| > threshold, без морфологии.

    Вход:  frames — список BGR uint8; warmup — число кадров для оценки фона.
    Выход: (masks — список (H, W) uint8 {0,1}; boxes — dict {frame: (x,y,w,h)|None}).
    """
    gray = [cv2.cvtColor(f, cv2.COLOR_BGR2GRAY) for f in frames]
    background = np.median(np.stack(gray[:warmup]), axis=0).astype(np.uint8)
    masks, boxes = [], {}
    for index, current in enumerate(gray):
        diff = cv2.absdiff(current, background)
        mask = (diff > threshold).astype(np.uint8)
        masks.append(mask)
        boxes[index] = largest_component_box(mask, min_area=min_area)
    return masks, boxes


tune_frames, tune_masks_gt, tune_gt = parts["tune"]

start = time.perf_counter()
baseline_masks, baseline_boxes = run_frame_difference(tune_frames, threshold=25)
elapsed = time.perf_counter() - start

box_metrics = evaluate_boxes(tune_gt, baseline_boxes)
mask_metrics = evaluate_masks(tune_masks_gt, baseline_masks)
log_run(segment="tune", method="frame_difference",
        params="threshold=25, warmup=15, morphology=none",
        fps=round(len(tune_frames) / elapsed, 1), seconds=round(elapsed, 3),
        seed=SEED, **box_metrics, **mask_metrics)

print("Baseline на настроечном фрагменте:", box_metrics, mask_metrics)
runs_table()

In [ ]:
# Визуальный контроль baseline: маска переднего плана и выделенный объект.
indices = [5, 25, 50, 70]
fig, axes = plt.subplots(2, len(indices), figsize=(3.2 * len(indices), 6))
for col, index in enumerate(indices):
    axes[0, col].imshow(cv2.cvtColor(tune_frames[index], cv2.COLOR_BGR2RGB))
    axes[0, col].set_title(f"кадр {index}", fontsize=9)
    axes[1, col].imshow(baseline_masks[index], cmap="gray")
    axes[1, col].set_title("маска (baseline)", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

show_frames(tune_frames, indices, baseline_boxes, "Объект внимания, baseline")

# Обратите внимание на отклик от колеблющейся помехи в левом нижнем углу:
# это источник ложных срабатываний, который должна снять либо многомодальная
# модель фона, либо морфология, либо критерий устойчивости.

## 8. Модели фона, морфология и объект внимания

Реализуйте единый интерфейс детектора переднего плана, чтобы разностный метод, MOG2 и KNN прогонялись одним и тем же кодом и сравнивались по одним и тем же метрикам.

Полезные вызовы: `cv2.createBackgroundSubtractorMOG2(history, varThreshold, detectShadows)`, `cv2.createBackgroundSubtractorKNN(history, dist2Threshold, detectShadows)`, метод `apply(frame, learningRate=...)`. При `detectShadows=True` тени кодируются значением 127 — их нужно обработать явно, иначе маска окажется трёхуровневой.

Морфология: `cv2.getStructuringElement`, `cv2.morphologyEx` с `cv2.MORPH_OPEN` и `cv2.MORPH_CLOSE`. Порядок операций и размер элемента фиксируйте и логируйте.

Устойчивость объекта внимания: одиночная наибольшая компонента в кадре неустойчива при коротких пропаданиях. Возможные критерии — сглаживание по нескольким кадрам, требование перекрытия с компонентой предыдущего кадра, минимальное время жизни компоненты.

In [ ]:
def detect_foreground(frames, method, *, morphology=None, min_area=150, **params):
    """Единый интерфейс выделения переднего плана и объекта внимания.

    Контракт.
    Вход:
        frames     — список кадров (H, W, 3) uint8 BGR в исходном порядке;
        method     — 'diff' | 'mog2' | 'knn';
        morphology — None или словарь вида
                     {'open': 3, 'close': 7, 'shape': cv2.MORPH_ELLIPSE};
        min_area   — минимальная площадь компоненты;
        params     — параметры модели фона (history, varThreshold,
                     dist2Threshold, learningRate, detectShadows, threshold ...).
    Выход:
        (masks, boxes): masks — список (H, W) uint8 {0, 1};
                        boxes — dict {frame_index: (x, y, w, h) или None}.

    Требования:
        1) кадры обрабатываются строго последовательно: модель фона зависит от
           истории, перемешивание порядка недопустимо;
        2) тени (значение 127 при detectShadows=True) обрабатываются явно;
        3) морфология применяется до выделения компонент;
        4) объект внимания выбирается по критерию устойчивости, а не только по
           площади в текущем кадре; критерий опишите в docstring своей функции.
    """
    raise NotImplementedError("TODO (задание 3.2, 3.3): реализуйте модели фона и постобработку")

In [ ]:
# TODO (задание 3.4): серия по параметрам НА НАСТРОЕЧНОМ ФРАГМЕНТЕ.
#
# Минимальный план (меняется один фактор за серию):
#   1) threshold разностного метода: [15, 25, 35, 50];
#   2) MOG2: varThreshold in [8, 16, 32, 64] при фиксированных history и learningRate;
#   3) MOG2: learningRate in [0.001, 0.005, 0.02, -1] при фиксированном varThreshold;
#   4) KNN: dist2Threshold in [200, 400, 800];
#   5) морфология: размер элемента открытия in [0, 3, 5, 7].
#
# Для каждой конфигурации логируйте: segment='tune', method, params, TP/FP/FN,
# precision, recall, f1, mean_iou, попиксельные метрики, время и fps.
#
# Отдельно ответьте измерением: какие конфигурации снимают отклик от помехи,
# и какой ценой в полноте это достигается.
#
# for var_threshold in [8, 16, 32, 64]:
#     masks, boxes = detect_foreground(tune_frames, "mog2", history=200,
#                                      varThreshold=var_threshold,
#                                      detectShadows=False,
#                                      morphology={"open": 3, "close": 7})
#     log_run(segment="tune", method="mog2", params=f"varThreshold={var_threshold}", ...)

runs_table()

In [ ]:
# TODO (задание 3.4): финальный прогон на ОЦЕНОЧНОМ фрагменте.
#
# Порядок:
#   1) по журналу настроечного фрагмента выберите одну конфигурацию для каждого
#      метода (разностный, MOG2, KNN) — критерий выбора укажите в отчёте;
#   2) прогоните ИХ ЖЕ, без дополнительной подстройки, на parts['eval'];
#   3) сравните с baseline на том же фрагменте;
#   4) отдельно посчитайте метрики до и после кадра со скачком освещения
#      (кадр 90 исходной последовательности) — это прямая проверка того, как
#      модель фона реагирует на изменение освещения.
#
# eval_frames, eval_masks_gt, eval_gt = parts["eval"]

pass

In [ ]:
# TODO (задание 3.1, отчётный артефакт): сохраните серию кадров с выделенным
# объектом и, при необходимости, видеофайл.
#
# annotated = []
# for index, frame in enumerate(eval_frames):
#     canvas = frame.copy()
#     box = final_boxes.get(index)
#     if box is not None:
#         x, y, w, h = box
#         cv2.rectangle(canvas, (x, y), (x + w, y + h), (0, 255, 0), 2)
#     annotated.append(canvas)
# save_video(annotated, OUTPUT_DIR / "detection.mp4", fps=25)

pass

## Отчёт

### Сводные таблицы

Обязательны:

1. «метод × конфигурация → precision, recall, F1, mean IoU, попиксельный IoU, fps» на настроечном фрагменте;
2. итоговая таблица на оценочном фрагменте: baseline против выбранных конфигураций MOG2 и KNN;
3. таблица «до/после скачка освещения».

In [ ]:
frame = runs_table()

if frame.empty:
    print("Журнал пуст: выполните разделы 7 и 8.")
else:
    columns = [c for c in ["segment", "method", "params", "precision", "recall", "f1",
                           "mean_iou", "px_iou", "FP", "FN", "fps"] if c in frame.columns]
    display(frame[columns].round(3))

    if {"precision", "recall", "method"} <= set(frame.columns):
        fig, ax = plt.subplots(figsize=(6.5, 5))
        for method, group in frame.groupby("method"):
            ax.scatter(group["recall"], group["precision"], label=method)
        ax.set_xlabel("полнота (recall)")
        ax.set_ylabel("точность (precision)")
        ax.set_xlim(-0.02, 1.02)
        ax.set_ylim(-0.02, 1.02)
        ax.set_title("компромисс пропуски/ложные срабатывания")
        ax.grid(alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.show()

save_runs()

### Выводы

**Наблюдения** (измеренные факты со ссылкой на строки журнала):

-

**Интерпретация** (чем объясняется рост ложных срабатываний или пропусков при изменении конкретного параметра; как повёл себя каждый метод на помехе и на скачке освещения):

-

**Выводы и их границы** (какое видео, какое разрешение, какие диапазоны параметров; что переносится на реальную съёмку, а что нет; какие сцены заведомо вне проверенных условий — движущаяся камера, множественные объекты, тени):

-

**Анализ ошибок**: приведите кадры с характерным пропуском и с характерным ложным срабатыванием, поясните причину каждого.

## Контрольные вопросы

Из [списка вопросов блока](README.md#контрольные-вопросы-блока):

6. Как модель фона MOG2 адаптируется к изменению освещения и чем это опасно?

Дополнительно:

- Чем разностный метод принципиально уступает многомодальной модели фона и в каких условиях эта разница исчезает?
- Почему параметры нельзя подбирать на том же фрагменте, на котором посчитаны итоговые метрики?
- Как изменится оценка, если объект внимания на несколько кадров останавливается?

## Чек-лист перед сдачей

- [ ] Ноутбук исполняется сверху вниз без ошибок после `Restart & Run All`.
- [ ] Указаны ФИО, группа, номер работы, источники данных (синтетика и/или реальное видео).
- [ ] Seed и версии библиотек зафиксированы и выведены.
- [ ] Организовано покадровое чтение и буферизация; порядок кадров не нарушен.
- [ ] Модель фона (MOG2 и/или KNN) сравнена с разностным методом на одних и тех же кадрах.
- [ ] Маски очищены морфологией, параметры морфологии зафиксированы и залогированы.
- [ ] Объект внимания выделяется по критерию устойчивости, критерий описан.
- [ ] Полнота и точность посчитаны на размеченном фрагменте, а не описаны словесно.
- [ ] Параметры подобраны на настроечном фрагменте, итоговые метрики посчитаны на оценочном.
- [ ] Есть серия кадров (или видео) с выделенным объектом.
- [ ] Есть сводные таблицы и график компромисса точность/полнота; журнал сохранён (`outputs_hw6/runs.csv`).
- [ ] Наблюдения, интерпретация и выводы разделены; указаны ограничения.